# 🏀 Multi-Camera Basketball Analysis
<a href="https://colab.research.google.com/github/video-db/videodb-cookbook/blob/main/real_time_streaming/multicam/Multicam_Basketball_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>
---

Welcome to the **Multi-Camera Basketball Analysis** notebook! Learn how to build a professional AI-powered basketball game analysis system with multi-camera coverage, real-time highlight detection via WebSocket, and automated multi-angle replay composition.

### 🎯 What You'll Build

A complete AI-powered multicam basketball analysis system that demonstrates:

- **👁️ SEE**: Connect to 3 cameras and preview live feeds
- **🧠 UNDERSTAND**: Index visual content across all cameras with AI analysis
- **🎬 ACT**: Detect events in real-time, receive WebSocket alerts, and create same-interval multi-angle videos

By the end, you'll have a working system that:
- Monitors 3 cameras simultaneously
- Detects specific events (slam dunks, three-pointers, fouls, fast breaks)
- Sends real-time alerts via WebSocket
- Creates same-interval multi-camera grid videos for replay review after camera clock alignment is confirmed

---

## 🛠 Setup & Installation

Let's start by installing the VideoDB Python SDK and connecting to your account.

---

### 📦 Install VideoDB

VideoDB is available as a [Python package](https://pypi.org/project/videodb). Run the cell below to install it.

In [ ]:
!pip install -q videodb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.9/95.9 kB 3.9 MB/s eta 0:00:00


### 🔗 Connect to VideoDB

You'll need an API key to interact with VideoDB. Provide it securely when prompted below.

> 💡 **Tip:** Get your free API key from the [VideoDB Console](https://console.videodb.io). Get $20 in free API credits upon sign-up — no credit card required!

In [ ]:
import videodb
import os
from getpass import getpass

api_key = getpass("Please enter your VideoDB API Key: ")
os.environ["VIDEO_DB_API_KEY"] = api_key

conn = videodb.connect()
coll = conn.get_collection()

print("✅ Connected to VideoDB successfully!")

Please enter your VideoDB API Key: ··········
✅ Connected to VideoDB successfully!


---

## 👁️ SEE - Connect Multi-Camera System

First, we'll connect to **3 cameras** monitoring a basketball arena from 3 camera angles. VideoDB handles RTSP stream ingestion seamlessly.

---

### 📹 Configure Cameras

In [ ]:
# Multi-camera configuration
CAMERA_CONFIG = {
    "cam1": {"name": "Main Court Field", "url": "rtsp://samples.rts.videodb.io:8554/bb-cam1"},
    "cam2": {"name": "North Basket Area", "url": "rtsp://samples.rts.videodb.io:8554/bb-cam2"},
    "cam3": {"name": "South Basket Area", "url": "rtsp://samples.rts.videodb.io:8554/bb-cam3"}
}

print("📋 Camera Configuration:")
for cam_id, info in CAMERA_CONFIG.items():
    print(f"   {cam_id}: {info['name']}")

📋 Camera Configuration:
   cam1: Main Court Field
   cam2: North Basket Area
   cam3: South Basket Area


---
### Connect all streams

In [ ]:
# Connect all cameras
print("🔌 Connecting to all cameras...\n")

streams = {}
for cam_id, cam_info in CAMERA_CONFIG.items():
    stream = coll.connect_rtstream(
        name=f"Basketball_{cam_id}",
        url=cam_info["url"],
        store=True,  # enables recording storage
    )
    streams[cam_id] = {"stream": stream, "info": cam_info}
    print(f"✅ {cam_id}: {stream.id}")

print(f"\n✅ All {len(streams)} cameras connected!")

🔌 Connecting to all cameras...

✅ cam1: rts-019fa917-4238-79a3-8662-54703ff9078b
✅ cam2: rts-019fa917-7540-7031-bd95-fc3f13d8ac47
✅ cam3: rts-019fa917-7e0c-7233-97ee-96244e9b92e9

✅ All 3 cameras connected!


#### To reconnect to existing stream:

In [ ]:
# EXISTING_STREAM_IDS = {
#     "cam1": "rts-xxxxx-xxxxx-xxxxx",  # Replace with your cam1 stream ID
#     "cam2": "rts-xxxxx-xxxxx-xxxxx",  # Replace with your cam2 stream ID
#     "cam3": "rts-xxxxx-xxxxx-xxxxx",  # Replace with your cam3 stream ID
# }

# print("🔌 Reconnecting to existing cameras...\n")

# streams = {}
# for cam_id, rtstream_id in EXISTING_STREAM_IDS.items():
#     stream = coll.get_rtstream(rtstream_id)
#     streams[cam_id] = {"stream": stream, "info": CAMERA_CONFIG[cam_id]}
#     print(f"✅ {cam_id}: {stream.id} ({CAMERA_CONFIG[cam_id]['name']})")

# print(f"\n✅ All {len(streams)} cameras reconnected!")

---

### 📺 Preview Live Feeds

Let's preview the last 2 minutes from each camera to verify connectivity:
> Wait for atleast 2 minuts before executing the following cell.

In [ ]:
from IPython.display import HTML
import time

# Get timestamps for last 2 minutes
now = int(time.time())
ten_seconds_ago = now - 120

print("📹 Generating preview streams for all cameras...\n")

# Generate player URLs and get stream URLs for the preview iframes
player_urls = {}
stream_urls = {}
video_titles = []
for cam_id, cam_data in streams.items():
    stream = cam_data["stream"]

    player_url = stream.generate_stream(ten_seconds_ago, now)
    stream_url = stream.stream_url

    player_urls[cam_id] = player_url
    stream_urls[cam_id] = stream_url
    video_titles.append(cam_data['info']['name'])
    print(f"✅ {cam_data['info']['name']}")
    print(f"  Player URL: {player_url}\n")

print("\n📺 Displaying all 3 camera feeds:\n")

# Create HTML with 3 videos in a row
html_content = """
<div style="display: grid; grid-template-columns: 1fr 1fr 1fr; gap: 10px; max-width: 900px;">
"""

for i, (cam_id, url) in enumerate(stream_urls.items()):
    html_content += f'''
        <div style="text-align: center;">
            <h4>{video_titles[i]}</h4>
            <iframe src="{url}" width="100%" height="200" frameborder="0" allowfullscreen></iframe>
        </div>
    '''

html_content += '</div>'

display(HTML(html_content))

📹 Generating preview streams for all cameras...

✅ Main Court Field
  Player URL: https://player.videodb.io/watch?v=wAc6Pn2B-a8

✅ North Basket Area
  Player URL: https://player.videodb.io/watch?v=MJtRDm1RnaM

✅ South Basket Area
  Player URL: https://player.videodb.io/watch?v=Z4H5ReHuJ_I


📺 Displaying all 3 camera feeds:



---

## 🧠 UNDERSTAND - AI-Powered Visual Analysis

Now we'll set up continuous AI-powered visual understanding and indexing across all 3 cameras. The AI will analyze frames every 10 seconds to detect basketball actions and plays.

---

### 🔍 Configure Visual Understanding

In [ ]:
# Continuous visual understanding configuration
VISUAL_UNDERSTANDING_CONFIG = {
    "segmentation": {
        "type": "time",
        "window": "10s"
    },
    "analyzer": {
        "type": "vlm",
        "name": "scene",
        "sampling": {"frame_count": 1},
        "config": {
            "prompt": """Analyze this basketball game footage. Describe:
            1. Slam dunks, three-point shots, and scoring plays
            2. Fast breaks and quick transition plays
            3. Fouls, physical contact, or player collisions
            Be specific about player positions and court location."""
        }
    }
}

print("📋 Visual Understanding Configuration:")
print(f"   Analyzing every {VISUAL_UNDERSTANDING_CONFIG['segmentation']['window']}")
print(f"   AI Focus: Basketball actions (dunks, shots, fast breaks, fouls)")

📋 Visual Understanding Configuration:
   Analyzing every 10s
   AI Focus: Basketball actions (dunks, shots, fast breaks, fouls)


In [ ]:
# Start one continuous visual understanding per camera
print("🔧 Creating visual understandings...\n")

scene_understandings = {}
for cam_id, cam_data in streams.items():
    stream = cam_data["stream"]
    understanding = stream.understand(
        segmentation=VISUAL_UNDERSTANDING_CONFIG["segmentation"],
        analyzers=[VISUAL_UNDERSTANDING_CONFIG["analyzer"]],
        store=True,
    )
    scene_output = understanding.outputs.get("scene")
    scene_understandings[cam_id] = {
        "understanding": understanding,
        "output": scene_output,
    }
    print(f"✅ {cam_id}: understanding started")

print(f"\n✅ Continuous visual understanding active on all {len(scene_understandings)} cameras!")

🔧 Creating visual understandings...

✅ cam1: understanding started
✅ cam2: understanding started
✅ cam3: understanding started

✅ Continuous visual understanding active on all 3 cameras!


### 🗂️ Index the Continuous Outputs

Create a semantic index for each camera's basketball-scene output.

In [ ]:
# Create one continuous scene index per camera
print("🔧 Creating visual indexes...\n")

scene_indexes = {}
for cam_id, cam_data in streams.items():
    stream = cam_data["stream"]
    understanding = scene_understandings[cam_id]["understanding"]
    scene_output = scene_understandings[cam_id]["output"]
    scene_index = stream.index(
        source=scene_output,
        name=f"Basketball_{cam_id}_Index",
        use_for=["semantic"],
    )
    scene_indexes[cam_id] = {
        "understanding": understanding,
        "understanding_id": understanding.id,
        "output": scene_output,
        "index": scene_index,
        "index_id": scene_index.id,
    }
    print(f"✅ {cam_id}: {scene_index.id}")

print(f"\n✅ Continuous visual indexing active on all {len(scene_indexes)} cameras!")

🔧 Creating visual indexes...

✅ cam1: idx-a98f30ac20a3b787
✅ cam2: idx-1a5dee13914a73ea
✅ cam3: idx-49bcc0fb7872acdd

✅ Continuous visual indexing active on all 3 cameras!


---

### 👀 Review Indexed Records

Let's inspect recent raw records from each camera's continuous index.

In [ ]:
import time
import json

for cam_id in streams.keys():
    scene_index = scene_indexes[cam_id]["index"]
    print(f"Reviewing 3 recent records for {cam_id} ({streams[cam_id]['info']['name']}):\n")

    crib_records_payload = scene_index.get_records(
        page=1,
        page_size=3,
    )

    # Process records to remove 'start' and 'end' fields
    processed_records_payload = crib_records_payload.copy()
    if "records" in processed_records_payload:
        new_records = []
        for record in processed_records_payload["records"]:
            new_record = record.copy()
            new_record.pop("start", None)
            new_record.pop("end", None)
            new_records.append(new_record)
        processed_records_payload["records"] = new_records

    print(json.dumps(processed_records_payload, indent=2, default=str))
    print("\n" + "-"*50 + "\n")

Reviewing 3 recent records for cam1 (Main Court Field):

{
  "next_page": true,
  "records": [
    {
      "description": "From this single frame, I can only give a limited analysis of the play in progress rather than the full game footage. Here\u2019s what is visible:\n\n### 1. Slam dunks, three-point shots, and scoring plays\n- No slam dunk is visible in this frame.\n- No shot attempt is clearly visible, so I can\u2019t confirm a three-point shot or made basket.\n- The offense appears to be set up in the **frontcourt near the right-side basket**. One white-uniformed player is positioned near the **right baseline/corner area**, while another is nearer the **free-throw lane/top of the paint**.\n- Several dark-uniformed defenders are clustered around the **paint and right side of the key**, suggesting a half-court scoring attempt or setup near the rim.\n\n### 2. Fast breaks and quick transition plays\n- This does **not** look like a fast break at this moment.\n- Most players are already

---

## 🎬 ACT - Event Detection & Multi-Angle Composition

Now comes the exciting part! We'll:
1. Create event detection rules
2. Monitor for alerts in real-time via WebSocket
3. Create same-interval multi-camera evidence videos after confirming the camera clocks are aligned

---

### 🚨 Define Events to Detect

In [ ]:
# Define 4 basketball events
EVENTS_CONFIG = [
    {"label": "slam_dunk", "prompt": "Detect when a player performs a slam dunk"},
    {"label": "three_pointer", "prompt": "Detect when a player scores a three-point shot"},
    {"label": "foul", "prompt": "Detect when a foul or physical contact occurs"},
    {"label": "fast_break", "prompt": "Detect a fast break or quick transition play"}
]

print("🎯 Creating event detection rules...\n")

events = {}
for cfg in EVENTS_CONFIG:
    event_id = conn.create_event(
        event_prompt=cfg["prompt"],
        label=cfg["label"]
    )
    events[cfg["label"]] = {"event_id": event_id}
    print(f"Event: {cfg['label']}")
    print(f"    ID: {event_id}")

print(f"\n✅ {len(events)} events ready for detection!")

🎯 Creating event detection rules...

Event: slam_dunk
    ID: c5bfc7a28922d064
Event: three_pointer
    ID: 79fc134bebc85ff5
Event: foul
    ID: 0d1b83d49a2baaba
Event: fast_break
    ID: c40ae5e386b1418d

✅ 4 events ready for detection!


---

### 🔌 Connect WebSocket for Real-Time Alerts

WebSockets let us receive alerts instantly as they happen:

In [ ]:
import asyncio

# Connect to WebSocket
ws_wrapper = conn.connect_websocket()
ws = await ws_wrapper.connect()

print(f"✅ WebSocket connected!")
print(f"   Connection ID: {ws.connection_id}")

INFO:videodb.websocket_client:WebSocket connected with ID: gVfJ1a4v_eO4KEhTYA==


✅ WebSocket connected!
   Connection ID: gVfJ1a4v_eO4KEhTYA==


Create alerts for every camera × event combination

In [ ]:
import os

RTSTREAM_ALERT_CALLBACK_URL = "https://example.com"
# Create alerts for all cameras × all events
print("🔔 Creating alerts...\n")

alerts = {}
for cam_id, idx_data in scene_indexes.items():
    alerts[cam_id] = {}
    print(f"--- Alerts for Camera: {cam_id} ---\n")
    for label, evt in events.items():
        alert_id = idx_data["index"].create_alert(
            evt["event_id"],
            callback_url=RTSTREAM_ALERT_CALLBACK_URL,
            ws_connection_id=ws.connection_id
        )
        alerts[cam_id][label] = alert_id
        print(f"Alert: {label}")
        print(f"    ID: {alert_id}")
    print(f"\n")

total_alerts = len(alerts) * len(events)
print(f"\n✅ {total_alerts} alerts active (3 cameras × 4 events)")



🔔 Creating alerts...

--- Alerts for Camera: cam1 ---

Alert: slam_dunk
    ID: 49b6bcd0f9ce2e41
Alert: three_pointer
    ID: 940545128b3f5f8c
Alert: foul
    ID: 81bb9f19d5dbd921
Alert: fast_break
    ID: 6455d682de002420


--- Alerts for Camera: cam2 ---

Alert: slam_dunk
    ID: f38d1cc7a08e0291
Alert: three_pointer
    ID: e0f176717535d998
Alert: foul
    ID: 4ee7ac4a0c64c879
Alert: fast_break
    ID: b5da3c3e31af893f


--- Alerts for Camera: cam3 ---

Alert: slam_dunk
    ID: 67710101c1df59cc
Alert: three_pointer
    ID: f987cecc5e7f14ef
Alert: foul
    ID: 380269c670caf86f
Alert: fast_break
    ID: a5218b1458ef85dd



✅ 12 alerts active (3 cameras × 4 events)


---

### 👂 Listen for Alerts

Let's listen for 30 seconds. When events are detected, we'll receive instant notifications:

In [ ]:
import json

# Create mapping from rtstream_id to cam info for easy lookup
rtstream_to_cam = {
    streams[cam_id]["stream"].id: {"cam_id": cam_id, "cam_name": CAMERA_CONFIG[cam_id]["name"]}
    for cam_id in streams.keys()
}

# Store alerts for later analysis
received_alerts = []

def get_alert_data(alert):
    if not isinstance(alert, dict):
        return {}
    data = alert.get("data") or alert
    return data if isinstance(data, dict) else {}

async def listen_for_alerts():
    timeout = 30  # Adjustable
    print(f"Listening for alerts ({timeout} seconds)...")
    print("The basketball analysis system will trigger alerts when events are detected\n")

    try:
        async with asyncio.timeout(timeout):
            async for msg in ws.receive():
                if not isinstance(msg, dict):
                    print("WebSocket message:", repr(msg))
                    continue
                if msg.get("channel") == "alert":
                    received_alerts.append(msg)
                    print(f"\nALERT #{len(received_alerts)} RECEIVED!")
                    print(json.dumps(msg, indent=2, default=str))
    except asyncio.TimeoutError:
        print(f"\nListening complete! {len(received_alerts)} alert(s) received")

# Start listening
await listen_for_alerts()

Listening for alerts (30 seconds)...
The basketball analysis system will trigger alerts when events are detected


ALERT #1 RECEIVED!
{
  "channel": "alert",
  "timestamp": "2026-07-28T14:48:11.732527+00:00",
  "rtstream_id": "rts-019fa917-4238-79a3-8662-54703ff9078b",
  "rtstream_name": "Basketball_cam1",
  "data": {
    "event_id": "alert-81bb9f19d5dbd921",
    "label": "foul",
    "triggered": true,
    "confidence": 0.86,
    "start": 1785250070.4139009,
    "end": 1785250081.4139152,
    "player_url": "https://console.videodb.io/player?url=https://rt.stream.videodb.io/manifests/rts-019fa917-4238-79a3-8662-54703ff9078b/1785250070000000-1785250082000000.m3u8",
    "stream_url": "https://rt.stream.videodb.io/manifests/rts-019fa917-4238-79a3-8662-54703ff9078b/1785250070000000-1785250082000000.m3u8",
    "explanation": "The frame shows clear body pressure and likely physical contact near the left lane/baseline, with the ballhandler closely defended by multiple maroon players. While no 

---

### 📊 Analyze Alert Metrics

In [ ]:
# Count alerts by event type
by_type = {}
for alert in received_alerts:
    label = get_alert_data(alert).get("label", "unknown")
    by_type[label] = by_type.get(label, 0) + 1

print("📊 Alerts by Event Type:")
print("-" * 50)
for label, count in sorted(by_type.items(), key=lambda x: x[1], reverse=True):
    # Format label: slam_dunk -> Slam Dunk
    formatted_label = label.replace("_", " ").title()
    print(f"   {formatted_label:.<35} {count}")

# Count alerts by camera (using rtstream_to_cam mapping)
by_cam = {}
for alert in received_alerts:
    rtstream_id = alert.get("rtstream_id", "unknown") if isinstance(alert, dict) else "unknown"
    cam_info = rtstream_to_cam.get(rtstream_id, {"cam_id": "unknown", "cam_name": "Unknown"})
    cam_key = f"{cam_info['cam_id']} - {cam_info['cam_name']}"
    by_cam[cam_key] = by_cam.get(cam_key, 0) + 1

print("\n📹 Alerts by Camera:")
print("-" * 50)
for cam, count in sorted(by_cam.items(), key=lambda x: x[1], reverse=True):
    print(f"   {cam:.<35} {count}")

print(f"\n✅ Total: {len(received_alerts)} alerts received")

📊 Alerts by Event Type:
--------------------------------------------------
   Foul............................... 4

📹 Alerts by Camera:
--------------------------------------------------
   cam1 - Main Court Field............ 3
   cam3 - South Basket Area........... 1

✅ Total: 4 alerts received


---

### 🔍 Select Alert to Investigate

Let's investigate an alert and create a multi-angle evidence video:

In [ ]:
# Select the first available alert
selected = next(
    alert for alert in received_alerts
)
data = get_alert_data(selected)
start_time = data.get("start")
end_time = data.get("end")
print(f"🚨 Selected Alert: {data.get('label', 'N/A')}")
print(f"   Confidence: {data.get('confidence', 'N/A')}")
print(f"   Time: {selected.get('timestamp', 'N/A')}")

# Use videodb play_stream to play the selected alert's video
from videodb import play_stream
player_url = data.get("stream_url")
print(f"\n▶️ Playing stream for selected alert: {data.get('label', 'N/A')}")
play_stream(player_url)

🚨 Selected Alert: foul
   Confidence: 0.86
   Time: 2026-07-28T14:48:11.732527+00:00

▶️ Playing stream for selected alert: foul


---

### 🎥 Generate Same-Interval Multi-Camera Clips

We'll request the same Unix interval from all 3 cameras. Confirm that the source camera clocks are aligned before treating the clips as synchronized.

In [ ]:
# Add 10 seconds of context around the alert
PADDING = 10
clip_start = int(start_time - PADDING)
clip_end = int(end_time + PADDING)
clip_duration = clip_end - clip_start

print(f"⏱️ Alert Time Window: {start_time} → {end_time}")
print(f"📹 With padding: {clip_start} → {clip_end}\n")

# Generate the same requested interval for all cameras
stream_urls = {}
player_urls_map = {}
for cam_id, cam_data in streams.items():
    stream = cam_data["stream"]

    # Must run both (critical for RTStream)
    player_url = stream.generate_stream(clip_start, clip_end)
    stream_url = stream.stream_url

    stream_urls[cam_id] = stream_url
    player_urls_map[cam_id] = player_url

print(f"✅ Same requested interval generated for 3 cameras\n")
print("Verify camera clock alignment before treating these clips as synchronized.\n")

print("Player URLs for each camera:")
for cam_id, p_url in player_urls_map.items():
    cam_name = CAMERA_CONFIG[cam_id]["name"]
    print(f"{cam_id} : {cam_name}")
    print(f"    {p_url}")

⏱️ Alert Time Window: 1785250070.4139009 → 1785250081.4139152
📹 With padding: 1785250060 → 1785250091

✅ Same requested interval generated for 3 cameras

Verify camera clock alignment before treating these clips as synchronized.

Player URLs for each camera:
cam1 : Main Court Field
    https://player.videodb.io/watch?v=1G3-IL-ED9s
cam2 : North Basket Area
    https://player.videodb.io/watch?v=_6TZ4cGqLeA
cam3 : South Basket Area
    https://player.videodb.io/watch?v=JWRrFc1YoBg


---

### 💾 Download Clips for Composition

Since the Timeline Editor requires VideoDB media IDs, we'll download these clips using ffmpeg.

In [ ]:
import subprocess
import os

# Download all 3 clips using ffmpeg
print("📥 Downloading clips with ffmpeg...\n")

downloads = {}
for cam_id, stream_url in stream_urls.items():
    output_file = f"{cam_id}_clip.mp4"

    # Use ffmpeg to download HLS stream
    cmd = [
        'ffmpeg',
        '-y',  # Overwrite output file
        '-i', stream_url,  # Input HLS URL
        '-c', 'copy',  # Copy codec (no re-encoding)
        '-loglevel', 'error',  # Only show errors
        output_file
    ]

    print(f"   {cam_id}: Downloading...")
    try:
        subprocess.run(cmd, check=True, capture_output=True)
        downloads[cam_id] = {"name": output_file}
        print(f"   ✅ {cam_id}: Saved as {output_file}\n")
    except subprocess.CalledProcessError as e:
        print(f"   ❌ {cam_id}: Failed - {e.stderr.decode()}")
        downloads[cam_id] = {"name": None, "error": str(e)}

print(f"✅ Downloaded {len([d for d in downloads.values() if d.get('name')])} clips!")

📥 Downloading clips with ffmpeg...

   cam1: Downloading...
   ✅ cam1: Saved as cam1_clip.mp4

   cam2: Downloading...
   ✅ cam2: Saved as cam2_clip.mp4

   cam3: Downloading...
   ✅ cam3: Saved as cam3_clip.mp4

✅ Downloaded 3 clips!


#### 💾 Uploading back to VideoDB

In [ ]:
# Upload clips back to VideoDB
print("📤 Uploading clips to VideoDB...\n")

videos = {}
for cam_id, dl_info in downloads.items():
    video = coll.upload(file_path=dl_info["name"])
    videos[cam_id] = {"video": video, "video_id": video.id}
    print(f"✅ {cam_id}: {video.id}\n")

print("✅ All clips uploaded and ready for composition!")

📤 Uploading clips to VideoDB...

✅ cam1: m-z-019fa938-659d-71a1-b3a4-808373f7cdc9

✅ cam2: m-z-019fa938-91d7-73a0-9197-1025759918eb

✅ cam3: m-z-019fa938-c069-7d71-a802-e508e200f26c

✅ All clips uploaded and ready for composition!


---
### 🎬 Create 3-Camera Replay Grid

Now compose the 3 camera angles requested from the same Unix interval. The replay is time-aligned only when the source camera clocks are aligned:
- **Cam 1 (Main Court Field)**: Top-left
- **Cam 2 (North Basket Area)**: Top-right
- **Cam 3 (South Basket Area)**: Bottom-center

In [ ]:
from videodb.editor import (
    Timeline, Track, Clip, VideoAsset, Position, Offset, Fit,
    TextAsset, Font, Border, Shadow, Background, TextAlignment
)

# Get minimum duration
print("📏 Checking video durations...\n")
video_durations = {}
for cam_id, vid_info in videos.items():
    video = vid_info["video"]
    video_durations[cam_id] = video.length

final_duration = min(video_durations.values())

# Create timeline with medium gray background
timeline = Timeline(conn)
timeline.resolution = "1280x720"
timeline.background = "#404040"

# Camera positions: top-left, top-right, bottom-center
cam_configs = [
    {"cam_id": "cam1", "position": Position.top_left, "offset": Offset(x=0.03, y=0.025)},
    {"cam_id": "cam2", "position": Position.top_right, "offset": Offset(x=-0.03, y=0.025)},
    {"cam_id": "cam3", "position": Position.bottom, "offset": Offset(x=0, y=-0.025)},
]

print("🎬 Building multi-camera grid...\n")

# Add video tracks
for config in cam_configs:
    cam_id = config["cam_id"]
    clip = Clip(
        asset=VideoAsset(id=videos[cam_id]["video_id"]),
        duration=final_duration,
        fit=Fit.crop,
        position=config["position"],
        offset=config["offset"],
        scale=0.45,
    )
    track = Track()
    track.add_clip(0, clip)
    timeline.add_track(track)

# Label positions matching each camera
label_configs = [
    {"cam_id": "cam1", "offset": Offset(x=-0.355, y=-0.45)},
    {"cam_id": "cam2", "offset": Offset(x=0.135, y=-0.45)},
    {"cam_id": "cam3", "offset": Offset(x=-0.110, y=0.05)},
]

# Add camera labels
for config in label_configs:
    cam_id = config["cam_id"]
    cam_name = CAMERA_CONFIG[cam_id]["name"]

    label_text = TextAsset(
        text=f"{cam_id.upper()}: {cam_name}",
        font=Font(family="Clear Sans", size=24, color="#FFFFFF"),
        background=Background(
            color="#000000",
            height=50,
            width=300,
            text_alignment=TextAlignment.center,
        ),
    )

    label_clip = Clip(
        asset=label_text,
        duration=final_duration,
        offset=config["offset"],
    )

    track = Track()
    track.add_clip(0, label_clip)
    timeline.add_track(track)

print(f"✅ Multi-camera replay ready!\n")
print(f"   Resolution: 1280x720")
print(f"   Duration: {final_duration:.2f}s")
print(f"   Layout: Top row (Main Court + North Basket) + Bottom center (South Basket)")

📏 Checking video durations...

🎬 Building multi-camera grid...

✅ Multi-camera replay ready!

   Resolution: 1280x720
   Duration: 19.00s
   Layout: Top row (Main Court + North Basket) + Bottom center (South Basket)


In [ ]:
# Generate final multi-camera view
from videodb import play_stream

print("🎬 Generating final video...\n")

stream = timeline.generate_stream()

print("✅ Multi-camera grid ready!")
print(f"Stream: {stream}\n")

play_stream(stream)

🎬 Generating final video...

✅ Multi-camera grid ready!
Stream: https://play.videodb.io/v1/c64fd692-66a8-420f-8122-f10fe93d617d.m3u8



<div style="background-color: #d4edda; color: #155724; padding: 12px; border-left: 5px solid #28a745; border-radius: 4px;">
    <strong>🎉 Success!</strong> You've created a professional same-interval multi-camera basketball replay from 3 different angles.
</div>

---

## 🧹 Cleanup

When you're done, let's disconnect all resources:

In [ ]:
print("🧹 Cleaning up resources...\n")

# Disable all alerts
for cam_id, cam_alerts in alerts.items():
    for label, alert_id in cam_alerts.items():
        scene_indexes[cam_id]["index"].disable_alert(alert_id)

print("✅ All alerts disabled")

# Stop continuous indexes and understandings before disconnecting streams
for cam_id, idx_data in scene_indexes.items():
    idx_data["index"].stop()
    idx_data["understanding"].stop()

print("✅ All continuous jobs stopped")

# Stop all streams
for cam_id, cam_data in streams.items():
    cam_data["stream"].stop()

print("✅ All streams stopped")

# Close the WebSocket after all producers have stopped
await ws_wrapper.close()
print("✅ WebSocket closed")
print("\n🎉 Cleanup complete!")

🧹 Cleaning up resources...

✅ All alerts disabled
✅ All continuous jobs stopped
✅ All streams stopped
✅ WebSocket closed

🎉 Cleanup complete!


---

# 🏁 Conclusion: Multi-Camera Basketball Analysis

Congratulations! You've built a complete AI-powered multi-camera basketball analysis system.

### 🎯 What You Accomplished

**👁️ SEE:**
- ✅ Connected 3 RTSP camera streams
- ✅ Previewed live feeds from multiple angles

**🧠 UNDERSTAND:**
- ✅ Indexed visual content across all cameras with AI
- ✅ Reviewed scene descriptions from different angles

**🎬 ACT:**
- ✅ Created 4 event detection rules
- ✅ Monitored real-time alerts via WebSocket
- ✅ Analyzed alert metrics by type and camera
- ✅ Generated same-interval multi-camera clips with an explicit clock-alignment check
- ✅ Composed a professional multi-camera grid video

---

### 🚀 What's Next?

Ready to build more advanced sports analysis systems?

- 📖 **[VideoDB Documentation](https://docs.videodb.io)**: Complete API reference
- 🍳 **[VideoDB Cookbook](https://github.com/video-db/videodb-cookbook)**: More multicam examples
- 💬 **[Discord Community](https://discord.com/invite/py9P639jGz)**: Get help and share projects

---

**Build intelligent sports analysis systems with VideoDB — real-time AI for every game.** 🏀

